[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/060.%20The%20Warehouse%20Slotting%20Optimization%20Problem/P56-Tier-3_executed.ipynb)

---


# 56. The Safety Stock and Reorder Point \((Q,r)\) Policy Problem

## Tier 3: Particle Swarm Optimization

### Purpose and scope

This notebook uses Particle Swarm Optimization (PSO), a population-based stochastic search method, to find a low-cost order quantity \(Q\) and reorder point \(r\). The inputs describe annual and daily demand, lead-time uncertainty, inventory costs, and a target cycle service level. The outputs are a candidate policy, its estimated annual cost, its estimated service level, and diagnostic plots.

The model is a teaching approximation rather than a complete inventory simulation. It assumes normally distributed demand during lead time, uses an approximate annual stockout-cost calculation, and searches within specified bounds. It does not model time-varying demand, order overlap, lost-sales dynamics, or a detailed distribution of shortages. The service-level penalty is a modeling device for discouraging policies below the target; the target is not implemented as a hard constraint.

### How PSO searches the policy space

A particle is one candidate policy: its position is the two-dimensional vector \(x_i=(Q_i,r_i)\), and its velocity determines how that candidate moves through the search space. Each particle remembers its best position, while the swarm also shares the best position found by any particle. The method alternates between evaluating cost and updating these positions and memories.

For particle \(i\) at iteration \(t\), the update is

$$
v_i^{t+1}=w_t v_i^t+c_1 u_1(p_i-x_i^t)+c_2 u_2(g-x_i^t),
\qquad
x_i^{t+1}=x_i^t+v_i^{t+1}.
$$

Here, \(v_i^t\) is velocity, \(p_i\) is the particle's personal-best position, \(g\) is the swarm's global-best position, and \(u_1,u_2\) are independent random values drawn uniformly from \([0,1]\). The inertia weight \(w_t\) decreases linearly from \(w_{\max}\) to \(w_{\min}\); \(c_1\) and \(c_2\) control the cognitive and social influences. After each position update, \(Q\) and \(r\) are clipped to their search bounds.

The objective is the modeled annual cost, not a proof of global optimality. PSO is stochastic, so different runs can produce different candidate solutions. This notebook does not set a random seed; reproducible results require controlling the random-number generators before running the experiment. The stopping test checks whether the best cost has improved by less than a specified relative amount over ten iterations, or whether the iteration limit has been reached.

### Learning sequence

The notebook defines the problem inputs and swarm, specifies the cost and service-level calculations, implements the PSO updates, runs the CardioStabil example, and then examines the solution with visualizations and a swarm-size sensitivity analysis. A cross-method comparison is meaningful only when results for the other methods use the same demand, cost, service-level, and modeling assumptions.

In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully for Particle Swarm Optimization.")

## 1. Define the problem data and particle swarm

The annual demand is converted to average daily demand using a 365-day year. The search bounds restrict the order quantity and reorder point to the ranges explored by the swarm; they are modeling choices, not universal inventory rules.

The demand during lead time has mean \(\mu_{LT}=\bar d L\). Under the model's independence and normal-approximation assumptions, its variance is

$$
\sigma_{LT}^{2}=L\sigma_d^{2}+\bar d^{\,2}\sigma_L^{2},
$$

where \(\bar d\) is average daily demand, \(L\) is mean lead time, \(\sigma_d\) is the standard deviation of daily demand, and \(\sigma_L\) is the standard deviation of lead time. The target's normal quantile is displayed as a reference; the optimizer evaluates service at each candidate reorder point.

In [ ]:
@dataclass
class DemandParameters:
    """Demand and lead-time inputs for the inventory model."""

    annual_demand: float
    daily_std: float
    lead_time: float
    lead_time_std: float


@dataclass
class CostParameters:
    """Unit, holding, ordering, and stockout cost inputs."""

    unit_cost: float
    holding_rate: float
    ordering_cost: float
    stockout_cost: float


class PSOInventoryOptimizer:
    """Particle Swarm Optimization for an inventory (Q, r) policy."""

    def __init__(
        self,
        demand_params: DemandParameters,
        cost_params: CostParameters,
        service_level: float,
        swarm_size: int = 30,
        max_iterations: int = 100,
    ):
        if demand_params.annual_demand <= 0:
            raise ValueError("Annual demand must be positive.")
        if demand_params.daily_std < 0 or demand_params.lead_time_std < 0:
            raise ValueError("Demand and lead-time standard deviations cannot be negative.")
        if demand_params.lead_time < 0:
            raise ValueError("Mean lead time cannot be negative.")
        if not 0 < service_level < 1:
            raise ValueError("The target service level must be between 0 and 1.")
        if swarm_size < 1 or max_iterations < 1:
            raise ValueError("Swarm size and maximum iterations must be positive.")
        if min(
            cost_params.unit_cost,
            cost_params.holding_rate,
            cost_params.ordering_cost,
            cost_params.stockout_cost,
        ) < 0:
            raise ValueError("Cost parameters cannot be negative.")

        self.demand_params = demand_params
        self.cost_params = cost_params
        self.service_level = service_level
        self.swarm_size = swarm_size
        self.max_iterations = max_iterations

        self.daily_demand = demand_params.annual_demand / 365
        self.z_score = stats.norm.ppf(service_level)

        self.w_max, self.w_min = 0.9, 0.4
        self.c1, self.c2 = 2.0, 2.0

        self.q_bounds = [10.0, demand_params.annual_demand / 2]
        mean_lead_time_demand = self.daily_demand * demand_params.lead_time
        self.r_bounds = [mean_lead_time_demand, 3 * mean_lead_time_demand]
        if self.q_bounds[1] < self.q_bounds[0]:
            raise ValueError("Annual demand is too small for the specified Q search bounds.")
        if self.r_bounds[1] <= self.r_bounds[0]:
            raise ValueError("The specified lead time does not define a nonzero r search interval.")

        self.initialize_swarm()

    def initialize_swarm(self) -> None:
        """Create particles with positions, velocities, and best-so-far records."""
        self.particles = []

        for _ in range(self.swarm_size):
            q = random.uniform(*self.q_bounds)
            r = random.uniform(*self.r_bounds)
            position = np.array([q, r], dtype=float)

            velocity = np.array(
                [
                    random.uniform(-0.1 * self.q_bounds[1], 0.1 * self.q_bounds[1]),
                    random.uniform(-0.1 * self.r_bounds[1], 0.1 * self.r_bounds[1]),
                ],
                dtype=float,
            )
            self.particles.append(
                {
                    "position": position,
                    "velocity": velocity,
                    "best_position": position.copy(),
                    "best_fitness": float("inf"),
                    "fitness": float("inf"),
                }
            )

        self.global_best_position = np.array(
            [np.mean(self.q_bounds), np.mean(self.r_bounds)], dtype=float
        )
        self.global_best_fitness = float("inf")

        print(f"Swarm initialized: {self.swarm_size} particles")
        print(f"Q bounds: [{self.q_bounds[0]:.1f}, {self.q_bounds[1]:.1f}]")
        print(f"r bounds: [{self.r_bounds[0]:.1f}, {self.r_bounds[1]:.1f}]")

## 2. Model the service level and annual cost

For a reorder point \(r\), the modeled cycle service level is the probability that demand during lead time does not exceed \(r\). With the normal approximation, this is the cumulative distribution function evaluated at the standardized reorder point:

$$
SL(r)=\Phi\!\left(\frac{r-\mu_{LT}}{\sigma_{LT}}\right).
$$

The safety stock is \(SS=\max(0,r-\mu_{LT})\), and cycle stock is \(Q/2\). The objective combines annual holding cost, annual ordering cost, an approximate stockout cost, and a penalty for falling below the requested service level:

$$
\begin{aligned}
C(Q,r)={}&\left(\frac{Q}{2}+SS\right)c h
+\frac{D}{Q}K
+\left(1-SL(r)\right)\frac{D}{4}p_s\\
&+50000\max\left(0,SL_{\text{target}}-SL(r)\right).
\end{aligned}
$$

Here, \(D\) is annual demand, \(c\) is unit cost, \(h\) is the annual holding-rate fraction, \(K\) is ordering cost per order, and \(p_s\) is the stockout-cost parameter. The factor \(D/4\) is the model's approximation for the annual exposure used in its stockout-cost term; it is not a detailed expected-shortage calculation. The final term is a penalty, so a service target is encouraged by the objective rather than guaranteed by a hard constraint.

The code exposes the cost components separately so that the objective, diagnostic plot, and reported total use the same calculations.

In [ ]:
class PSOInventoryOptimizer(PSOInventoryOptimizer):
    """Add service-level and cost calculations to the optimizer."""

    def _calculate_service_level(self, r: float) -> float:
        """Evaluate the normal-approximation cycle service level at reorder point r."""
        expected_demand = self.daily_demand * self.demand_params.lead_time
        demand_variance = (
            self.demand_params.lead_time * self.demand_params.daily_std**2
        )
        lead_time_variance = (
            self.daily_demand**2 * self.demand_params.lead_time_std**2
        )
        combined_std = math.sqrt(demand_variance + lead_time_variance)

        if combined_std == 0:
            return 1.0 if r >= expected_demand else 0.0

        z_value = (r - expected_demand) / combined_std
        return float(stats.norm.cdf(z_value))

    def calculate_cost_components(self, q: float, r: float) -> Dict[str, float]:
        """Return the modeled annual cost components for a candidate policy."""
        q = max(1.0, float(q))
        r = max(1.0, float(r))

        expected_lead_demand = self.daily_demand * self.demand_params.lead_time
        safety_stock = max(0.0, r - expected_lead_demand)
        cycle_stock = q / 2
        holding_cost = (
            (cycle_stock + safety_stock)
            * self.cost_params.unit_cost
            * self.cost_params.holding_rate
        )
        ordering_cost = (
            self.demand_params.annual_demand / q
        ) * self.cost_params.ordering_cost

        achieved_service_level = self._calculate_service_level(r)
        stockout_probability = max(0.0, 1.0 - achieved_service_level)
        expected_stockouts = (
            stockout_probability * self.demand_params.annual_demand / 4
        )
        stockout_cost = expected_stockouts * self.cost_params.stockout_cost
        service_penalty = (
            max(0.0, self.service_level - achieved_service_level) * 50000
        )

        return {
            "holding_cost": holding_cost,
            "ordering_cost": ordering_cost,
            "stockout_cost": stockout_cost,
            "service_penalty": service_penalty,
            "safety_stock": safety_stock,
            "service_level_achieved": achieved_service_level,
        }

    def calculate_fitness(self, q: float, r: float) -> float:
        """Return total modeled annual cost; PSO minimizes this objective."""
        components = self.calculate_cost_components(q, r)
        return sum(
            components[name]
            for name in (
                "holding_cost",
                "ordering_cost",
                "stockout_cost",
                "service_penalty",
            )
        )


print("Fitness function and service-level calculation are ready.")

## 3. Update particles and run the optimization

At the start of the run, every particle is evaluated so that the personal and global bests refer to actual candidate costs. Each iteration then updates every particle, records the best cost found so far, and checks for stagnation. The best-so-far history cannot increase because a global best is replaced only when a lower cost is found.

The implementation's early-stopping rule compares the best cost with the value ten iterations earlier. If relative improvement is below 0.1%, the run stops; otherwise, it continues up to the specified iteration limit. This is a practical stopping rule, not a certificate that the global optimum has been found.

In [ ]:
class PSOInventoryOptimizer(PSOInventoryOptimizer):
    """Add particle updates and the main PSO search loop."""

    def update_particle(self, particle: Dict, iteration: int) -> None:
        """Apply the PSO velocity and position equations to one particle."""
        inertia = self.w_max - (
            (self.w_max - self.w_min) * iteration / self.max_iterations
        )
        random_cognitive = random.random()
        random_social = random.random()

        cognitive = self.c1 * random_cognitive * (
            particle["best_position"] - particle["position"]
        )
        social = self.c2 * random_social * (
            self.global_best_position - particle["position"]
        )

        particle["velocity"] = (
            inertia * particle["velocity"] + cognitive + social
        )
        particle["position"] += particle["velocity"]

        # Clipping keeps every candidate inside the declared search domain.
        particle["position"][0] = np.clip(
            particle["position"][0], *self.q_bounds
        )
        particle["position"][1] = np.clip(
            particle["position"][1], *self.r_bounds
        )

        particle["fitness"] = self.calculate_fitness(
            particle["position"][0], particle["position"][1]
        )

        if particle["fitness"] < particle["best_fitness"]:
            particle["best_fitness"] = particle["fitness"]
            particle["best_position"] = particle["position"].copy()

        if particle["fitness"] < self.global_best_fitness:
            self.global_best_fitness = particle["fitness"]
            self.global_best_position = particle["position"].copy()

    def optimize(self) -> Dict:
        """Run PSO and return the best policy and its convergence history."""
        fitness_history: List[float] = []

        print("Initial evaluation of swarm...")
        for particle in self.particles:
            particle["fitness"] = self.calculate_fitness(
                particle["position"][0], particle["position"][1]
            )
            if particle["fitness"] < particle["best_fitness"]:
                particle["best_fitness"] = particle["fitness"]
                particle["best_position"] = particle["position"].copy()
            if particle["fitness"] < self.global_best_fitness:
                self.global_best_fitness = particle["fitness"]
                self.global_best_position = particle["position"].copy()

        fitness_history.append(self.global_best_fitness)
        print("Starting PSO optimization...")

        stopped_early = False
        iterations_run = 0

        for iteration in range(self.max_iterations):
            for particle in self.particles:
                self.update_particle(particle, iteration)

            fitness_history.append(self.global_best_fitness)
            iterations_run = iteration + 1

            if len(fitness_history) >= 11:
                cost_ten_iterations_ago = fitness_history[-11]
                current_best_cost = fitness_history[-1]
                denominator = max(abs(cost_ten_iterations_ago), 1e-12)
                relative_improvement = (
                    cost_ten_iterations_ago - current_best_cost
                ) / denominator
                if relative_improvement < 0.001:
                    stopped_early = True
                    print(f"Early stopping at iteration {iterations_run}.")
                    break

            if iterations_run % 20 == 0:
                print(
                    f"Iteration {iterations_run}: "
                    f"Best annual cost = ${self.global_best_fitness:,.2f}"
                )

        optimal_q, optimal_r = self.global_best_position
        components = self.calculate_cost_components(optimal_q, optimal_r)
        total_cost = self.calculate_fitness(optimal_q, optimal_r)

        if not math.isclose(
            total_cost,
            self.global_best_fitness,
            rel_tol=1e-10,
            abs_tol=1e-8,
        ):
            raise RuntimeError("The reported best cost does not match its policy cost.")

        return {
            "Q": float(optimal_q),
            "r": float(optimal_r),
            "safety_stock": components["safety_stock"],
            "total_cost": total_cost,
            "service_level_achieved": components["service_level_achieved"],
            "cost_components": components,
            "fitness_history": fitness_history,
            "iterations": iterations_run,
            "converged": stopped_early,
        }


print("Particle update and PSO optimization loop are ready.")

## 4. Run the CardioStabil example

The following inputs define the example instance. PSO returns a stochastic candidate solution within its search bounds. Interpret its service level and cost under the approximations stated above; the reported candidate is not a guarantee of global optimality.

In [ ]:
demand_params = DemandParameters(
    annual_demand=12000,
    daily_std=10,
    lead_time=14,
    lead_time_std=3,
)

cost_params = CostParameters(
    unit_cost=450,
    holding_rate=0.25,
    ordering_cost=200,
    stockout_cost=2000,
)

service_level = 0.995

print("=== INITIALIZING PSO OPTIMIZER ===")
optimizer = PSOInventoryOptimizer(
    demand_params=demand_params,
    cost_params=cost_params,
    service_level=service_level,
    swarm_size=40,
    max_iterations=150,
)

print(f"\nTarget cycle service level: {service_level * 100:.1f}%")
print(f"Average daily demand: {optimizer.daily_demand:.2f} units")
print(f"Target normal quantile: {optimizer.z_score:.3f}")

print("\n=== RUNNING PSO OPTIMIZATION ===")
result = optimizer.optimize()

print("\n=== PSO OPTIMIZATION RESULTS ===")
print(f"Stopped early by stagnation test: {result['converged']}")
print(f"Iterations completed: {result['iterations']}")
print(f"Order quantity, Q: {result['Q']:.2f} units")
print(f"Reorder point, r: {result['r']:.2f} units")
print(f"Safety stock: {result['safety_stock']:.2f} units")
print(f"Modeled annual cost: ${result['total_cost']:,.2f}")
print(
    "Achieved cycle service level: "
    f"{result['service_level_achieved']:.4f} "
    f"({result['service_level_achieved'] * 100:.2f}%)"
)
print(f"Target cycle service level: {service_level:.4f} ({service_level * 100:.2f}%)")
print(
    "Service-level gap: "
    f"{(result['service_level_achieved'] - service_level) * 100:+.3f} "
    "percentage points"
)

## 5. Inspect convergence, candidate positions, service, and costs

The convergence panel plots the best cost recorded at each evaluation; a flat tail indicates that the swarm has not found a better candidate during that interval, not that global optimality has been proved. The position panel shows the final location of each particle and identifies the best policy found. Color represents normalized final particle fitness, where lower modeled cost corresponds to better fitness. The remaining panels compare target and achieved service and show every modeled cost component, including the service penalty.

In [ ]:
def create_pso_visualization(result: Dict, optimizer: PSOInventoryOptimizer) -> None:
    """Plot convergence, final particle positions, service, and cost components."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(15, 12)
    )
    fig.suptitle(
        "Particle Swarm Optimization: CardioStabil Inventory Policy",
        fontsize=16,
        fontweight="bold",
    )

    fitness_history = result["fitness_history"]
    ax1.plot(
        range(len(fitness_history)),
        fitness_history,
        color="blue",
        linewidth=2,
        label="Best modeled annual cost",
    )
    ax1.set_xlabel("Evaluation index (initial evaluation is 0)")
    ax1.set_ylabel("Modeled annual cost ($)")
    ax1.set_title("Best-Cost Convergence History")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    q_positions = np.array([p["position"][0] for p in optimizer.particles])
    r_positions = np.array([p["position"][1] for p in optimizer.particles])
    fitnesses = np.array([p["fitness"] for p in optimizer.particles])
    fitness_range = np.ptp(fitnesses)
    normalized_fitnesses = (
        (fitnesses - fitnesses.min()) / fitness_range
        if fitness_range > 0
        else np.zeros_like(fitnesses)
    )

    scatter = ax2.scatter(
        q_positions,
        r_positions,
        c=normalized_fitnesses,
        cmap="viridis",
        alpha=0.75,
        s=60,
        edgecolors="black",
    )
    ax2.scatter(
        result["Q"],
        result["r"],
        color="red",
        s=200,
        marker="*",
        edgecolors="black",
        linewidth=1,
        label="Global best",
        zorder=5,
    )
    ax2.set_xlabel("Order quantity, Q (units)")
    ax2.set_ylabel("Reorder point, r (units)")
    ax2.set_title("Final Particle Positions")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    colorbar = fig.colorbar(scatter, ax=ax2)
    colorbar.set_label("Normalized final fitness (lower is better)")

    target_percent = optimizer.service_level * 100
    achieved_percent = result["service_level_achieved"] * 100
    service_bars = ax3.bar(
        ["Target", "Achieved"],
        [target_percent, achieved_percent],
        color=["lightblue", "lightgreen"],
        edgecolor="black",
    )
    ax3.set_ylabel("Cycle service level (%)")
    ax3.set_title("Target and Achieved Service Level")
    ax3.set_ylim(0, 105)
    ax3.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(service_bars, [target_percent, achieved_percent]):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{value:.2f}%",
            ha="center",
            va="bottom",
        )
    gap = achieved_percent - target_percent
    ax3.text(
        0.5,
        0.08,
        f"Gap: {gap:+.3f} percentage points",
        transform=ax3.transAxes,
        ha="center",
    )

    components = result["cost_components"]
    component_names = [
        "Holding",
        "Ordering",
        "Stockout",
        "Service penalty",
    ]
    component_values = [
        components["holding_cost"],
        components["ordering_cost"],
        components["stockout_cost"],
        components["service_penalty"],
    ]
    if sum(component_values) > 0:
        ax4.pie(
            component_values,
            labels=component_names,
            autopct="%1.1f%%",
            startangle=90,
        )
    else:
        ax4.text(0.5, 0.5, "All modeled cost components are zero.", ha="center")
        ax4.set_axis_off()
    ax4.set_title(f"Cost Components\nTotal: ${result['total_cost']:,.2f}")

    plt.tight_layout()
    plt.show()


create_pso_visualization(result, optimizer)

## 6. Cross-method comparison

Cost or policy comparisons across methods are interpretable only when every method uses this same problem instance and the same objective. No matching-instance outputs for the mathematical formulation or priority-based heuristic are supplied here, so the notebook does not infer comparative costs or claim that one method is superior. The function below can display the PSO result alongside reference results supplied by the reader; reference rows must contain results computed under the same assumptions.

In [ ]:
def compare_with_reference_methods(
    pso_result: Dict,
    reference_results: Optional[List[Dict]] = None,
) -> pd.DataFrame:
    """Tabulate PSO and optional same-instance reference-method results."""
    pso_row = {
        "method": "Particle Swarm Optimization",
        "Q": pso_result["Q"],
        "r": pso_result["r"],
        "safety_stock": pso_result["safety_stock"],
        "total_cost": pso_result["total_cost"],
        "service_level": pso_result["service_level_achieved"],
    }
    comparison_rows = [pso_row]

    if reference_results:
        required_fields = set(pso_row)
        for reference in reference_results:
            missing_fields = required_fields.difference(reference)
            if missing_fields:
                raise ValueError(
                    "Each reference result must contain: "
                    + ", ".join(sorted(missing_fields))
                )
            comparison_rows.append(dict(reference))
    else:
        print(
            "No same-instance reference results were supplied; "
            "comparative improvements are not calculated."
        )

    comparison_df = pd.DataFrame(comparison_rows)
    print(comparison_df.round(4).to_string(index=False))
    return comparison_df


comparison_df = compare_with_reference_methods(result)

## 7. Sensitivity to swarm size

Swarm size changes how many candidate policies are explored in each iteration. This experiment compares the resulting candidate cost, service level, policy, and stopping behavior for several swarm sizes while holding the inventory instance and iteration limit fixed. Because the algorithm is stochastic and no seed is set, the results are individual runs, not estimates of average performance across repeated trials. The main example's optimizer and result remain unchanged because each experiment creates a separate optimizer.

In [ ]:
def parameter_sensitivity_analysis(
    demand_params: DemandParameters,
    cost_params: CostParameters,
    service_level: float,
) -> pd.DataFrame:
    """Examine how swarm size affects one stochastic run of PSO."""
    swarm_sizes = [20, 40, 60]
    results = []

    for swarm_size in swarm_sizes:
        print(f"\nTesting swarm size: {swarm_size}")
        test_optimizer = PSOInventoryOptimizer(
            demand_params=demand_params,
            cost_params=cost_params,
            service_level=service_level,
            swarm_size=swarm_size,
            max_iterations=100,
        )
        test_result = test_optimizer.optimize()
        results.append(
            {
                "Swarm_Size": swarm_size,
                "Total_Cost": test_result["total_cost"],
                "Iterations": test_result["iterations"],
                "Stopped_Early": test_result["converged"],
                "Service_Level": test_result["service_level_achieved"],
                "Q": test_result["Q"],
                "r": test_result["r"],
            }
        )
        print(f"  Modeled annual cost: ${test_result['total_cost']:,.2f}")
        print(f"  Iterations completed: {test_result['iterations']}")
        print(f"  Stopped early by stagnation test: {test_result['converged']}")

    sensitivity_df = pd.DataFrame(results)
    print("\n=== SWARM-SIZE SENSITIVITY RESULTS ===")
    print(sensitivity_df.round(4).to_string(index=False))

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(15, 10)
    )
    fig.suptitle("PSO Sensitivity to Swarm Size", fontsize=14, fontweight="bold")

    ax1.plot(
        sensitivity_df["Swarm_Size"],
        sensitivity_df["Total_Cost"],
        "bo-",
        linewidth=2,
        markersize=8,
    )
    ax1.set_xlabel("Swarm size (particles)")
    ax1.set_ylabel("Modeled annual cost ($)")
    ax1.set_title("Candidate Cost by Swarm Size")
    ax1.grid(True, alpha=0.3)

    ax2.plot(
        sensitivity_df["Swarm_Size"],
        sensitivity_df["Iterations"],
        "ro-",
        linewidth=2,
        markersize=8,
    )
    ax2.set_xlabel("Swarm size (particles)")
    ax2.set_ylabel("Iterations completed")
    ax2.set_title("Iterations by Swarm Size")
    ax2.grid(True, alpha=0.3)

    ax3.plot(
        sensitivity_df["Swarm_Size"],
        sensitivity_df["Service_Level"] * 100,
        "go-",
        linewidth=2,
        markersize=8,
        label="Achieved",
    )
    ax3.axhline(
        y=service_level * 100,
        color="red",
        linestyle="--",
        label=f"Target ({service_level * 100:.2f}%)",
    )
    ax3.set_xlabel("Swarm size (particles)")
    ax3.set_ylabel("Cycle service level (%)")
    ax3.set_title("Service Level by Swarm Size")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    for _, row in sensitivity_df.iterrows():
        ax4.scatter(
            row["Q"],
            row["r"],
            s=100,
            alpha=0.75,
            label=f"Swarm size {int(row['Swarm_Size'])}",
            edgecolors="black",
        )
    ax4.set_xlabel("Order quantity, Q (units)")
    ax4.set_ylabel("Reorder point, r (units)")
    ax4.set_title("Candidate Policies by Swarm Size")
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return sensitivity_df


sensitivity_results = parameter_sensitivity_analysis(
    demand_params=demand_params,
    cost_params=cost_params,
    service_level=service_level,
)

## 8. Summarize the run

The final summary reports the configured PSO parameters and the candidate policy returned by the main run. “Stopped early” means only that the specified stagnation test was met; reaching the iteration limit or passing that test does not establish global optimality. Interpret any cost comparison only when the reference result uses the same instance and cost model.

In [ ]:
print("=== PARTICLE SWARM OPTIMIZATION SUMMARY ===")
print("\nALGORITHM SETTINGS:")
print(f"  Swarm size: {optimizer.swarm_size} particles")
print(f"  Maximum iterations: {optimizer.max_iterations}")
print(f"  Iterations completed: {result['iterations']}")
print(f"  Stopped early by stagnation test: {result['converged']}")
print(f"  Inertia weight range: {optimizer.w_max:.1f} to {optimizer.w_min:.1f}")
print(f"  Cognitive coefficient: {optimizer.c1:.1f}")
print(f"  Social coefficient: {optimizer.c2:.1f}")

print("\nBEST POLICY FOUND:")
print(f"  Order quantity, Q: {result['Q']:.2f} units")
print(f"  Reorder point, r: {result['r']:.2f} units")
print(f"  Safety stock: {result['safety_stock']:.2f} units")
print(f"  Modeled annual cost: ${result['total_cost']:,.2f}")
print(
    "  Achieved cycle service level: "
    f"{result['service_level_achieved'] * 100:.3f}%"
)
print(f"  Target cycle service level: {service_level * 100:.3f}%")

print("\nCOST COMPONENTS:")
for component_name in (
    "holding_cost",
    "ordering_cost",
    "stockout_cost",
    "service_penalty",
):
    print(
        f"  {component_name.replace('_', ' ').title()}: "
        f"${result['cost_components'][component_name]:,.2f}"
    )

print("\nInterpretation:")
print(
    "  The policy is the best candidate recorded by this stochastic PSO run "
    "within the specified search bounds."
)
print(
    "  Its service and cost estimates follow the normal-approximation and "
    "cost assumptions defined in this notebook."
)